# Notebook 9: Model Comparison, Cross-Validation & Conclusions
## Federal Reserve Interest Rate Prediction

**This notebook:**
1. Loads all saved model results
2. Produces comprehensive comparison charts
3. Analyzes overfitting/underfitting across all models
4. Demonstrates TimeSeriesSplit vs Standard K-Fold CV
5. Summarizes key findings and limitations


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"

import json, pickle
from sklearn.model_selection import (cross_val_score, learning_curve,
                                     TimeSeriesSplit, KFold)
from sklearn.preprocessing import StandardScaler


In [ ]:
with open(f"{OUT_PATH}/results/regression_results.json")    as f: reg_res  = json.load(f)
with open(f"{OUT_PATH}/results/classification_results.json") as f: cls_res  = json.load(f)
with open(f"{OUT_PATH}/results/clustering_results.json")     as f: clust    = json.load(f)
with open(f"{OUT_PATH}/results/arm_results.json")            as f: arm      = json.load(f)
with open(f"{OUT_PATH}/results/pca_results.json")            as f: pca      = json.load(f)
with open(f"{OUT_PATH}/results/statistical_analysis.json")   as f: stat     = json.load(f)
print("All results loaded.")


## 1. Regression Performance Comparison

In [ ]:
reg_df = pd.DataFrame({m: {'RMSE':r['RMSE'],'MAE':r['MAE'],'R2':r['R2'],
                             'CV R2':r['cv']['mean'],'Fit':r.get('learning_curve',{}).get('status','—')}
                   for m,r in reg_res.items()}).T.sort_values('R2', ascending=False)
display(reg_df.style.highlight_max(subset=['R2','CV R2'], color='lightgreen')
                     .highlight_min(subset=['RMSE','MAE'], color='lightgreen'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_r = reg_df.index
for ax, col, title in zip(axes, ['R2','RMSE','MAE'],
                           ['R² Score (higher=better)','RMSE (lower=better)','MAE (lower=better)']):
    bars = ax.bar(models_r, reg_df[col], color=PALETTE[:len(models_r)])
    ax.set_xticklabels(models_r, rotation=35, ha='right')
    ax.set_title(title, fontweight='bold')
    for bar, v in zip(bars, reg_df[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f'{v:.3f}', ha='center', fontsize=8)
fig.suptitle('Regression Models — Complete Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 2. Classification Performance Comparison

In [ ]:
cls_df = pd.DataFrame({m: {'Accuracy':r['Accuracy'],'F1':r['F1'],
                              'Precision':r['Precision'],'Recall':r['Recall'],
                              'CV Acc':r['cv']['mean']}
                   for m,r in cls_res.items()}).T.sort_values('Accuracy', ascending=False)
display(cls_df.style.highlight_max(color='lightgreen'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models_c = cls_df.index
for ax, col, title in zip(axes, ['Accuracy','F1'],
                           ['Accuracy (higher=better)','F1 Score (higher=better)']):
    bars = ax.bar(models_c, cls_df[col], color=PALETTE[:len(models_c)])
    ax.axhline(1/3, color='red', linestyle='--', alpha=0.7, label='Random baseline (33.3%)')
    ax.set_xticklabels(models_c, rotation=35, ha='right')
    ax.set_title(title, fontweight='bold'); ax.set_ylim(0,1.1); ax.legend()
    for bar, v in zip(bars, cls_df[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', fontsize=8)
fig.suptitle('Classification Models — Complete Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 3. TimeSeriesSplit vs Standard K-Fold: Why It Matters

In [ ]:
with open(f"{OUT_PATH}/results/preprocessed_data.pkl", "rb") as f:
    data = pickle.load(f)
X_scaled = data['X_scaled']; y_reg = data['y_reg']; y_cls = data['y_cls']

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
gbr = GradientBoostingRegressor(n_estimators=50, random_state=42)
gbc = GradientBoostingClassifier(n_estimators=50, random_state=42)

tscv = TimeSeriesSplit(n_splits=5)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

print("Gradient Boosting Regressor:")
ts_r2  = cross_val_score(gbr, X_scaled, y_reg, cv=tscv,  scoring='r2').mean()
kf_r2  = cross_val_score(gbr, X_scaled, y_reg, cv=kfold, scoring='r2').mean()
print(f"  TimeSeriesSplit R²: {ts_r2:.4f} (correct for time series)")
print(f"  Standard K-Fold R²: {kf_r2:.4f} (INFLATED due to leakage)")
print(f"  Inflation: {kf_r2 - ts_r2:.4f}")

print("\nGradient Boosting Classifier:")
ts_acc  = cross_val_score(gbc, X_scaled, y_cls, cv=tscv,  scoring='accuracy').mean()
kf_acc  = cross_val_score(gbc, X_scaled, y_cls, cv=kfold, scoring='accuracy').mean()
print(f"  TimeSeriesSplit Acc: {ts_acc:.4f} (correct for time series)")
print(f"  Standard K-Fold Acc: {kf_acc:.4f} (INFLATED due to leakage)")
print(f"  Inflation: {kf_acc - ts_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, ts, kf, title in [(axes[0], ts_r2, kf_r2, 'Regression (R²)'),
                           (axes[1], ts_acc, kf_acc, 'Classification (Acc)')]:
    bars = ax.bar(['TimeSeriesSplit\n(Correct)','Standard K-Fold\n(Leakage)'],
                  [ts, kf], color=['#4CAF50','#F44336'])
    for bar, v in zip(bars, [ts, kf]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_title(f'{title} CV Comparison', fontweight='bold')
    ax.set_ylim(0, 1.1)
fig.suptitle('Why TimeSeriesSplit Matters: CV Score Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 4. Overfitting/Underfitting Summary

In [ ]:
oc_data = {m: r['learning_curve'] for m,r in reg_res.items() if 'learning_curve' in r}
oc_data.update({m+'(C)': r['learning_curve'] for m,r in cls_res.items() if 'learning_curve' in r})

labels_oc = list(oc_data.keys())
train_s = [oc_data[k]['train_score'] for k in labels_oc]
val_s   = [oc_data[k]['val_score']   for k in labels_oc]
gaps    = [oc_data[k]['gap']         for k in labels_oc]
status  = [oc_data[k]['status']      for k in labels_oc]

x = np.arange(len(labels_oc))
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
bars_tr = axes[0].bar(x-0.2, train_s, 0.35, label='Train', color='#2196F3', alpha=0.8)
bars_vl = axes[0].bar(x+0.2, val_s,   0.35, label='Validation', color='#F44336', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels_oc, rotation=45, ha='right', fontsize=8)
axes[0].set_title('Train vs Validation Score — All Models', fontweight='bold')
axes[0].legend(); axes[0].set_ylim(0, 1.1)
for i, (g, s) in enumerate(zip(gaps, status)):
    col = '#4CAF50' if s=='GOOD FIT' else ('#FF9800' if s=='OVERFITTING' else '#F44336')
    axes[0].text(i, max(train_s[i], val_s[i]) + 0.02, s[:4], ha='center', fontsize=7, color=col)

colors_gap = ['#4CAF50' if g < 0.05 else ('#FF9800' if g < 0.15 else '#F44336') for g in gaps]
axes[1].bar(labels_oc, gaps, color=colors_gap)
axes[1].axhline(0.05, color='orange', linestyle='--', alpha=0.7, label='Overfitting threshold (0.05)')
axes[1].axhline(0.15, color='red', linestyle='--', alpha=0.7, label='Severe overfitting (0.15)')
axes[1].set_xticklabels(labels_oc, rotation=45, ha='right', fontsize=8)
axes[1].set_title('Train-Validation Gap (Overfitting Indicator)', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()


## 5. Key Findings & Conclusions

In [ ]:
print("=" * 70)
print("COMPLETE ANALYSIS SUMMARY")
print("=" * 70)

best_reg = max(reg_res, key=lambda x: reg_res[x]['R2'])
best_cls = max(cls_res, key=lambda x: cls_res[x]['Accuracy'])

print(f"\n REGRESSION BEST: {best_reg}")
print(f"   R² = {reg_res[best_reg]['R2']}")
print(f"   RMSE = {reg_res[best_reg]['RMSE']}")

print(f"\n CLASSIFICATION BEST: {best_cls}")
print(f"   Accuracy = {cls_res[best_cls]['Accuracy']:.1%}")
print(f"   F1 Score = {cls_res[best_cls]['F1']}")
print(f"   Improvement over random: +{(cls_res[best_cls]['Accuracy'] - 0.333)*100:.1f}pp")

print(f"\n PCA")
print(f"   4 components explain 95.7% of variance")
print(f"   PC1={pca['explained_variance_ratio'][0]*100:.1f}% | PC2={pca['explained_variance_ratio'][1]*100:.1f}% | PC3={pca['explained_variance_ratio'][2]*100:.1f}%")

print(f"\n KEY STATISTICAL FINDINGS")
ttest = stat['ttest_inflation']
print(f"   T-Test: High inflation -> {ttest['high_mean']:.2f}% vs {ttest['low_mean']:.2f}% rates")
print(f"   Spearman rho (Inflation): {stat['spearman']['InflationConsumerPrice']['rho']}")
print(f"   All features: Non-normal (Shapiro-Wilk)")

print(f"\n CLUSTERING")
print(f"   KMeans: {clust['kmeans']['best_k']} clusters (Silhouette={clust['kmeans']['best_silhouette']:.4f})")
print(f"   DBSCAN: {clust['dbscan']['n_clusters']} clusters, {clust['dbscan']['noise_pct']:.1f}% noise")

print(f"\n ASSOCIATION RULES")
print(f"   {arm['n_frequent_itemsets']} frequent itemsets, {arm['n_rules']} rules")
print(f"   Max lift: {arm['top_lift']:.4f} (GDP_Mid => RealGDP_Mid)")


In [ ]:
# Final radar chart comparison for top models
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

# Simple bar comparison: top models across all dimensions
fig, ax = plt.subplots(figsize=(12, 6))
top_reg = sorted(reg_res.items(), key=lambda x: x[1]['R2'], reverse=True)[:4]
top_cls = sorted(cls_res.items(), key=lambda x: x[1]['Accuracy'], reverse=True)[:4]

all_top = [(f"{m} (Reg)", r['R2']) for m,r in top_reg] +           [(f"{m} (Cls)", r['Accuracy']) for m,r in top_cls]

labels_t = [x[0] for x in all_top]
scores_t = [x[1] for x in all_top]
colors_t = ['#2196F3']*4 + ['#F44336']*4

bars_t = ax.bar(labels_t, scores_t, color=colors_t)
ax.set_xticklabels(labels_t, rotation=35, ha='right')
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Top Models: Regression (R²) vs Classification (Accuracy)', fontweight='bold')
for bar, v in zip(bars_t, scores_t):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
handles = [mpatches.Patch(color='#2196F3', label='Regression R²'),
           mpatches.Patch(color='#F44336', label='Classification Accuracy')]
ax.legend(handles=handles)
plt.tight_layout(); plt.show()
